# Feature Engineering for Training Ready Data in SageMaker

# 1. Apply transformations to convert raw data into training-ready features

# 1(a) Load Dataset from GitHub

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
# Replace with your actual GitHub raw URL
url = "https://raw.githubusercontent.com/pluralsight-cloud/Lab-Feature-Engineering-for-Training-Ready-Data-in-SageMaker/refs/heads/main/raw_customer_data.csv"
df = pd.read_csv(url)
df.head()

# 1(b) Explore Dataset

In [ ]:
# Structure
print("Shape:", df.shape)
print("\nColumns:\n", df.columns)
# Data types
print("\nData Types:\n", df.dtypes)
# Summary stats
df.describe(include='all')

# 1(c) Encode Categorical Variables

In [ ]:
# Standardize gender first
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])
df['gender'] = df['gender'].replace({'M': 'Male', 'F': 'Female'})
# Encode gender
df['gender_encoded'] = df['gender'].map({'Male': 1, 'Female': 0})

# Encode product category (One-hot encoding)
# Standardize BEFORE encoding
df['product_category'] = df['product_category'].str.lower()
# Then encode
df = pd.get_dummies(df, columns=['product_category'])

# Convert boolean to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

le = LabelEncoder()
df['city_encoded'] = le.fit_transform(df['city'])

df.head()

In [ ]:
# Drop original columns after encoding
df = df.drop(columns=['gender', 'city'])
df.head()

# 1(d) Create Derived Features

In [ ]:
# Convert date
df['purchase_date'] = pd.to_datetime(df['purchase_date'])

# Extract month
df['purchase_month'] = df['purchase_date'].dt.month

# Create ratio feature
df['purchase_amount'] = df['purchase_amount'].fillna(df['purchase_amount'].median())
df['income_per_purchase'] = df['income'] / df['purchase_amount']


df.head()

# 2. Refine feature values for consistency and modeling readiness

# 2(a) Remove Duplicate Records

In [ ]:
initial_rows = df.shape[0]

#df = df.drop_duplicates(subset=df.columns.difference(['income_per_purchase']))
df = df.drop_duplicates()
print("Duplicates removed:", initial_rows - df.shape[0])
df.head()

# 2(b) Handle Missing Values

In [ ]:
missing_before = df.isnull().sum()

print("Missing values BEFORE handling:\n", missing_before)


In [ ]:
# Age → median
df['age'] = df['age'].fillna(df['age'].median())
missing_after = df.isnull().sum()

print("\nMissing values AFTER handling:\n", missing_after)

In [ ]:
# convert age to integer
df['age'] = df['age'].astype(int)
df.head()

# 2(c) Create Target Variable (Purchased)

In [ ]:
df['purchased'] = (df['is_returned'] == 0).astype(int)
df.head()

# 2(d) Remove Data Leakage Columns

In [ ]:
# Drop columns that shouldn't be used in training
# is_returned is literally used to CREATE the target
# Model can “cheat” and get perfect accuracy
cols_to_drop = ['is_returned']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

df.head()

In [ ]:
# Remove other columns that is NOT required for prediction of target variable
cols_to_drop = ['customer_id', 'purchase_date', ]
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

df.head()

# 3.	Validate the feature set for data quality and modeling risks

# 3(a)	Check for duplicate records in the dataset

In [ ]:
print("\n Checking for duplicate records...")

duplicate_count = df.duplicated().sum()

print(f" Number of duplicate records: {duplicate_count}")

if duplicate_count > 0:
    print(" Duplicate records found!")
else:
    print(" No duplicate records found.")

# 3(b) 	Check for missing values 

In [ ]:
print("\n Checking for missing values...")

missing_values = df.isnull().sum()

print(missing_values)

total_missing = missing_values.sum()

if total_missing > 0:
    print(f"\n Total missing values: {total_missing}")
else:
    print("\n No missing values found.")

# 3(c) Evaluate Class Balance of Target Variable

In [ ]:
print("\n Evaluating class balance for target variable...")

class_counts = df['purchased'].value_counts()
class_percent = df['purchased'].value_counts(normalize=True) * 100

print("\nClass Distribution (Count):")
print(class_counts)

print("\nClass Distribution (%):")
print(class_percent)

# Simple interpretation
if class_percent.min() < 20:
    print("\n Imbalanced dataset detected!")
else:
    print("\n Dataset is reasonably balanced.")

# 3(d). Document Feature Meaning & Transformation Logic

In [ ]:
print("\n Feature Documentation:\n")

print("""
customer_id:
- Unique identifier for each customer
- Removed to avoid overfitting

age:
- Customer age
- Missing values filled using median
- Converted to integer

gender_encoded:
- Encoded gender (Male=1, Female=0)
- Original 'gender' column removed

city_encoded:
- One-hot encoded city features
- Standardized to lowercase before encoding

income:
- Annual income of customer
- Used as numerical feature

purchase_amount:
- Amount spent by customer
- Missing values filled using median

income_per_purchase:
- Derived feature = income / purchase_amount
- Represents spending capacity

product_category_*:
- One-hot encoded product categories
- Cleaned for consistent casing before encoding

purchase_month:
- Extracted from purchase_date
- Helps capture seasonality

purchased (target variable):
- Derived using is_returned
- 1 = Purchased (not returned)
- 0 = Not purchased (returned)

Removed Columns:
- customer_id (identifier)
- purchase_date (raw date, replaced with derived features)
- is_returned (used to derive target → leakage)
""")